# SIH26027: Exploratory Data Analysis (EDA)
## AI-Powered Automatic Block Planning for Indian Railways
**Problem Statement**: Predict Maintenance Task Urgency & Risk Scoring
**Target**: `urgent` (0 = Routine / 1 = Urgent Priority Block Required)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({'font.sans-serif': 'Arial', 'figure.autolayout': True})

df = pd.read_csv('../data/processed/railway_maintenance_tasks.csv')
print(f"Loaded dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()

### 1. Dataset Overview & Data Quality

In [ ]:
print("Missing Values:")
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"Unique tasks: {df['task_id'].nunique():,}, Unique assets: {df['asset_id'].nunique():,}")
df.describe().T

### 2. Target Variable Analysis (`urgent`)

In [ ]:
print("Urgency Class Counts:")
print(df['urgent'].value_counts())
print("\nUrgency Class Percentages:")
print(df['urgent'].value_counts(normalize=True) * 100)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x='urgent', palette=['#3b82f6', '#ef4444'], ax=ax[0])
ax[0].set_title('Urgent Counts (0 vs 1)')
ax[1].pie(df['urgent'].value_counts(), labels=['0: Routine (64.5%)', '1: Urgent (35.5%)'],
         autopct='%1.1f%%', colors=['#3b82f6', '#ef4444'], explode=(0, 0.05))
ax[1].set_title('Urgent Proportions')
plt.show()

### 3. Categorical Features vs Target Urgency

In [ ]:
for col in ['defect_severity', 'asset_criticality', 'operational_impact', 'department']:
    res = df.groupby(col)['urgent'].agg(['count', 'mean']).rename(columns={'mean': 'urgency_rate'})
    res['urgency_rate'] *= 100
    print(f"\n--- {col} vs Urgency ---:")
    print(res.round(2))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.countplot(data=df, x='defect_severity', hue='urgent', order=['LOW', 'MEDIUM', 'HIGH', 'CRITICAL'], ax=axes[0, 0])
sns.countplot(data=df, x='asset_criticality', hue='urgent', order=['LOW', 'MEDIUM', 'HIGH', 'CRITICAL'], ax=axes[0, 1])
sns.countplot(data=df, x='operational_impact', hue='urgent', order=['LOW', 'MEDIUM', 'HIGH', 'CRITICAL'], ax=axes[1, 0])
sns.countplot(data=df, x='department', hue='urgent', ax=axes[1, 1])
plt.tight_layout()
plt.show()

### 4. Numerical Features vs Target Urgency

In [ ]:
num_cols = ['asset_age_years', 'num_open_defects', 'days_since_defect', 'days_overdue',
            'days_since_last_maintenance', 'previous_failures', 'failures_last_12_months',
            'trains_per_day', 'goods_trains_per_day', 'maintenance_duration_hours']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
bp_cols = ['days_overdue', 'days_since_defect', 'failures_last_12_months', 'trains_per_day', 'num_open_defects', 'asset_age_years']
for i, c in enumerate(bp_cols):
    sns.boxplot(data=df, x='urgent', y=c, palette=['#3b82f6', '#ef4444'], ax=axes[i])
plt.tight_layout()
plt.show()

### 5. Correlation Analysis & Leakage Check

In [ ]:
corr = df[num_cols + ['urgent']].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-0.4, vmax=1.0)
plt.title('Pearson Correlation Heatmap')
plt.show()

print("Top Correlations with urgent:")
print(corr['urgent'].sort_values(ascending=False))